# Day 14: Production Data Pipeline Architecture

**Objective:** Build a robust, production-ready data engineering pipeline that integrates:
- Configuration management using YAML (`config.yaml`)
- Centralized structured logging (`logs/`)
- Incremental data ingestion with state tracking (`output/manifest.json`)
- Data quality validation with rejection thresholds (`validator.py`)

## 1. Imports and Setup
Import required standard library and third-party modules.

In [ ]:
import os
import sys
import json
import logging
from datetime import datetime
from pathlib import Path
import yaml
import pandas as pd

from validator import validate_price

# Set working directory to day-14 task folder
CURRENT_DIR = Path.cwd().resolve()
TASK_DIR = CURRENT_DIR if CURRENT_DIR.name == "day-14 task" else CURRENT_DIR / "day-14 task"
if str(TASK_DIR) not in sys.path:
    sys.path.insert(0, str(TASK_DIR))

print(f"Working Directory: {TASK_DIR}")

## 2. Load Configuration (`config.yaml`)
Load pipeline configuration specifying paths, rejection thresholds, numeric mappings, date fields, and allowed statuses.

In [ ]:
CONFIG_PATH = TASK_DIR / "config.yaml"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

print("Loaded Configuration:")
print(yaml.dump(config, default_flow_style=False))

## 3. Configure Logging
Configure structured pipeline logging writing to both console and timestamped log files in the `logs/` directory.

In [ ]:
log_dir = TASK_DIR / config["paths"]["log_folder"]
log_dir.mkdir(parents=True, exist_ok=True)

log_filename = f"pipeline_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.log"
log_filepath = log_dir / log_filename

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(log_filepath, encoding="utf-8"),
        logging.StreamHandler(sys.stdout),
    ],
    force=True,
)

logger = logging.getLogger("ProductionPipeline")
logger.info("Logging initialized successfully.")

## 4. Manifest Management (`output/manifest.json`)
Manage manifest state to enable idempotent, incremental processing and avoid reprocessing already ingested files.

In [ ]:
output_dir = TASK_DIR / config["paths"]["output_folder"]
output_dir.mkdir(parents=True, exist_ok=True)
manifest_path = output_dir / "manifest.json"


def load_manifest(path: Path) -> dict:
    """Load the processing manifest or initialize if not found."""
    if path.exists() and path.stat().st_size > 0:
        with open(path, "r", encoding="utf-8") as file:
            return json.load(file)
    return {"processed_files": []}


def update_manifest(path: Path, filename: str):
    """Record newly processed file into the manifest."""
    manifest = load_manifest(path)
    if filename not in manifest["processed_files"]:
        manifest["processed_files"].append(filename)
        with open(path, "w", encoding="utf-8") as file:
            json.dump(manifest, file, indent=4)
        logger.info(f"Manifest updated: Added {filename}")


manifest = load_manifest(manifest_path)
print(f"Already processed files: {manifest['processed_files']}")

## 5. Data Quality Validation and Processing
Define validation logic enforcing required schema, allowed order status values, numeric constraints, and rejection thresholds.

In [ ]:
def validate_dataset(df: pd.DataFrame, config: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Validate dataframe rows and partition into valid and rejected records."""
    allowed_statuses = set(config["mappings"]["allowed_order_status"])
    rejection_threshold = config["validation"]["rejection_threshold"]

    valid_mask = pd.Series(True, index=df.index)

    # Validate allowed order status if column exists
    if "order_status" in df.columns:
        status_mask = df["order_status"].isin(allowed_statuses)
        valid_mask &= status_mask

    # Validate price if column exists
    if "price" in df.columns:
        price_mask = df["price"].apply(lambda p: validate_price(p) if pd.notna(p) else False)
        valid_mask &= price_mask

    valid_df = df[valid_mask].copy()
    rejected_df = df[~valid_mask].copy()

    rejection_rate = (len(rejected_df) / len(df)) * 100 if len(df) > 0 else 0.0
    logger.info(f"Records: Total={len(df)}, Valid={len(valid_df)}, Rejected={len(rejected_df)} ({rejection_rate:.2f}%)")

    if rejection_rate > rejection_threshold:
        logger.warning(f"Rejection rate ({rejection_rate:.2f}%) exceeded threshold ({rejection_threshold}%)!")

    return valid_df, rejected_df

## 6. End-to-End Pipeline Execution
Run the full pipeline: discover incoming files, check manifest, validate records, write outputs, and update manifest.

In [ ]:
def run_pipeline():
    logger.info("========== PIPELINE STARTED ==========")
    input_dir = TASK_DIR / config["paths"]["input_folder"]
    input_dir.mkdir(parents=True, exist_ok=True)

    csv_files = list(input_dir.glob("*.csv"))
    logger.info(f"Found {len(csv_files)} CSV files in {input_dir}")

    manifest = load_manifest(manifest_path)

    for csv_file in csv_files:
        if csv_file.name in manifest["processed_files"]:
            logger.info(f"Skipping already processed file: {csv_file.name}")
            continue

        logger.info(f"Processing file: {csv_file.name}")
        df = pd.read_csv(csv_file)
        valid_df, rejected_df = validate_dataset(df, config)

        # Save processed valid data
        output_file = output_dir / f"clean_{csv_file.name}"
        valid_df.to_csv(output_file, index=False)

        # Record in manifest
        update_manifest(manifest_path, csv_file.name)

    logger.info("========== PIPELINE COMPLETED ==========")


run_pipeline()